# Cleaning & Transforming Data

The Silver layer is where raw, messy Bronze data becomes trustworthy. This notebook covers the core transformation toolkit: deduplication, null handling, safe type casting, string cleaning, date parsing, conditional logic, and the quarantine pattern for bad records.

In this notebook we'll cover:
- Deduplication — `DISTINCT`, `dropDuplicates`, and window-based dedup
- Null handling — filtering, filling, and the null vs empty string trap
- Type casting — `CAST` vs `TRY_CAST` and why it matters
- String cleaning — trim, regex, split, concat
- Date and timestamp parsing
- Conditional logic — `CASE WHEN`, `IF`, `NULLIF`, `COALESCE`
- The quarantine pattern — routing bad records without losing them

In [ ]:
# Setup — raw Bronze data with realistic quality issues
spark.sql("CREATE SCHEMA IF NOT EXISTS silver_demo")
spark.sql("USE SCHEMA silver_demo")

spark.sql("""
  CREATE OR REPLACE TABLE bronze_orders (
    order_id    STRING,
    customer_id STRING,
    amount      STRING,   -- raw string; may be 'N/A', '', or negative
    order_date  STRING,   -- raw string; mixed formats
    region      STRING,
    email       STRING
  ) USING DELTA
""")

spark.sql("""
  INSERT INTO bronze_orders VALUES
    ('ORD001', 'CUST01', '120.50',  '2024-01-15',  ' UK ',  'alice@example.com'),
    ('ORD002', 'CUST02', '340.00',  '15/01/2024',  'US',    'BOB@EXAMPLE.COM'),
    ('ORD003', 'CUST01', 'N/A',     '2024-01-16',  'UK',    NULL),
    ('ORD004', 'CUST03', '-50.00',  '2024-01-16',  'DE',    'carol@example.com'),
    ('ORD001', 'CUST01', '120.50',  '2024-01-15',  ' UK ',  'alice@example.com'),  -- duplicate
    ('ORD005', 'CUST02', '',        '2024-01-17',  'US',    'bob@example.com'),
    ('ORD006', NULL,     '75.00',   '2024-01-17',  'FR',    'dave@example.com')
""")

spark.sql("SELECT * FROM bronze_orders").show(truncate=False)

## Deduplication

### DISTINCT / dropDuplicates — Full-Row Dedup

Removes rows where **all** column values are identical.

```sql
SELECT DISTINCT * FROM bronze_orders;
```

```python
df.dropDuplicates()            # all columns
df.dropDuplicates(["order_id"]) # subset — keeps first occurrence
```

### Window-Based Dedup — Deterministic

`dropDuplicates` keeps an arbitrary row when there are duplicates — which row is kept is non-deterministic. When you need to keep the **latest** or **earliest** record, use a window function:

```sql
SELECT * FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date DESC) AS rn
  FROM   bronze_orders
) WHERE rn = 1;
```

`ROW_NUMBER()` assigns 1 to the most recent row per `order_id`. Filtering `rn = 1` keeps exactly one row per key, deterministically.

In [ ]:
# Full-row dedup — removes exact duplicate rows
deduped = spark.sql("SELECT DISTINCT * FROM bronze_orders")
print(f"Before: {spark.sql('SELECT COUNT(*) FROM bronze_orders').collect()[0][0]} rows")
print(f"After DISTINCT: {deduped.count()} rows")

In [ ]:
# Window-based dedup — keep latest row per order_id
spark.sql("""
  SELECT * EXCEPT(rn) FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY order_date DESC) AS rn
    FROM bronze_orders
  ) WHERE rn = 1
""").show(truncate=False)

## Null Handling

### NULL vs Empty String

These are not the same in Spark. `''` is a valid string value; `NULL` means the value is absent. Your cleaning logic must handle both:

```sql
-- Wrong: misses empty strings
WHERE amount IS NOT NULL

-- Correct: catches both null and empty
WHERE amount IS NOT NULL AND amount != ''

-- Or: convert empty string to null first, then filter
WHERE NULLIF(amount, '') IS NOT NULL
```

### COALESCE — First Non-Null Value

```sql
-- Use backup_email if primary email is null
SELECT COALESCE(email, backup_email, 'unknown@example.com') AS email
FROM   customers;
```

### fillna / fill — Replace Nulls

```python
df.fillna("unknown", subset=["region"])    # string columns
df.fillna(0.0,       subset=["amount"])    # numeric columns
df.fillna({"region": "unknown", "amount": 0.0})  # per-column map
```

### dropna — Remove Rows with Nulls

```python
df.dropna()                        # any null in any column
df.dropna(subset=["customer_id"])  # null only in specified columns
df.dropna(how="all")               # only rows where ALL columns are null
```

In [ ]:
from pyspark.sql import functions as F

df = spark.table("bronze_orders")

# NULLIF converts empty strings to null, then dropna removes them
df_no_empty = df.withColumn("amount",   F.nullif(F.col("amount"),   F.lit("")))\
                .withColumn("customer_id", F.nullif(F.col("customer_id"), F.lit("")))

# Drop rows missing business-critical fields
df_required = df_no_empty.dropna(subset=["order_id", "customer_id", "amount"])
print(f"Rows after dropping missing required fields: {df_required.count()}")
df_required.show(truncate=False)

## Type Casting — CAST vs TRY_CAST

`CAST` throws an error if the value cannot be converted. `TRY_CAST` returns `null` instead — much safer for raw data.

```sql
-- CAST fails hard on 'N/A'
SELECT CAST('N/A' AS DOUBLE);   -- SparkException

-- TRY_CAST returns null gracefully
SELECT TRY_CAST('N/A' AS DOUBLE);   -- NULL
SELECT TRY_CAST('120.50' AS DOUBLE); -- 120.5
```

> **Exam tip:** In Silver layer transformations on raw string data, always prefer `TRY_CAST` over `CAST`. After the cast, filter out the nulls (or route them to quarantine). Using `CAST` on unvalidated input is a common source of pipeline failures.

```sql
-- Safe Silver pattern
SELECT
  order_id,
  TRY_CAST(amount     AS DOUBLE)    AS amount,
  TRY_CAST(order_date AS DATE)      AS order_date
FROM bronze_orders
WHERE TRY_CAST(amount AS DOUBLE) > 0   -- also filters 'N/A', '', negatives
```

In [ ]:
# TRY_CAST — nullifies unconvertible values instead of failing
spark.sql("""
  SELECT
    order_id,
    amount                              AS raw_amount,
    TRY_CAST(amount AS DOUBLE)          AS amount_double,
    order_date                          AS raw_date,
    TRY_CAST(order_date AS DATE)        AS order_date_typed
  FROM bronze_orders
""").show(truncate=False)

## String Cleaning

```sql
-- Trim whitespace (very common with CSV/manual data entry)
TRIM(region)                -- removes leading and trailing spaces
LTRIM(region)               -- leading only
RTRIM(region)               -- trailing only

-- Case normalisation
LOWER(email)                -- 'BOB@EXAMPLE.COM' → 'bob@example.com'
UPPER(region)

-- Regex
REGEXP_REPLACE(phone, '[^0-9]', '')        -- strip non-digits
REGEXP_EXTRACT(email, '(@.+)$', 1)         -- extract domain

-- Split and access
SPLIT(full_name, ' ')[0]    -- first word (first name)
SPLIT(full_name, ' ')[1]    -- second word (last name)

-- Concatenate
CONCAT(first_name, ' ', last_name)
CONCAT_WS(', ', city, country)   -- join with separator, skips nulls

-- Length checks for validation
LENGTH(order_id) = 6        -- validate fixed-length IDs
```

In [ ]:
# String cleaning — trim region, lowercase email
spark.sql("""
  SELECT
    order_id,
    region                  AS raw_region,
    TRIM(region)            AS region_clean,
    email                   AS raw_email,
    LOWER(TRIM(email))      AS email_clean
  FROM bronze_orders
  WHERE email IS NOT NULL
""").show(truncate=False)

## Date and Timestamp Parsing

Raw data rarely arrives in a consistent date format. Two functions cover most cases:

```sql
-- Parse a string into DATE using a format pattern
TO_DATE('15/01/2024', 'dd/MM/yyyy')    -- DATE: 2024-01-15
TO_DATE('2024-01-15', 'yyyy-MM-dd')    -- DATE: 2024-01-15

-- Parse a string into TIMESTAMP
TO_TIMESTAMP('2024-01-15 09:30:00', 'yyyy-MM-dd HH:mm:ss')

-- Extract parts
YEAR(order_date)     -- 2024
MONTH(order_date)    -- 1
DAYOFWEEK(order_date)-- 1=Sunday … 7=Saturday

-- Arithmetic
DATEDIFF(end_date, start_date)      -- days between dates
DATE_ADD(order_date, 7)             -- 7 days later
DATE_TRUNC('month', order_date)     -- first day of the month

-- Current values
CURRENT_DATE()       -- today as DATE
CURRENT_TIMESTAMP()  -- now as TIMESTAMP
```

> When source dates arrive in **multiple formats** (e.g., some rows `yyyy-MM-dd`, others `dd/MM/yyyy`), use `COALESCE(TRY_TO_DATE(..., fmt1), TRY_TO_DATE(..., fmt2))` to try each format in order.

In [ ]:
# Handle mixed date formats with COALESCE + TRY_CAST
spark.sql("""
  SELECT
    order_id,
    order_date AS raw_date,
    COALESCE(
      TRY_TO_DATE(order_date, 'yyyy-MM-dd'),
      TRY_TO_DATE(order_date, 'dd/MM/yyyy')
    ) AS order_date_clean
  FROM bronze_orders
""").show(truncate=False)

## Conditional Logic

### CASE WHEN

The most flexible conditional expression — works in SQL and is translated to the same thing as `when()` in PySpark.

```sql
SELECT
  order_id,
  amount,
  CASE
    WHEN amount >= 500 THEN 'high'
    WHEN amount >= 100 THEN 'medium'
    ELSE                    'low'
  END AS order_tier
FROM silver_orders;
```

### Inline Helpers

```sql
IF(amount > 0, amount, NULL)      -- IF(condition, true_val, false_val)

NULLIF(amount, 0)                 -- returns NULL if amount = 0, else amount
                                  -- useful: NULLIF avoids divide-by-zero

IFNULL(email, 'unknown')         -- returns 'unknown' if email IS NULL

COALESCE(a, b, c)                -- first non-null among a, b, c
```

### PySpark Equivalent

```python
from pyspark.sql import functions as F

df.withColumn("order_tier",
    F.when(F.col("amount") >= 500, "high")
     .when(F.col("amount") >= 100, "medium")
     .otherwise("low")
)

In [ ]:
# Conditional: classify orders by tier after casting amount
spark.sql("""
  SELECT
    order_id,
    TRY_CAST(amount AS DOUBLE) AS amount,
    CASE
      WHEN TRY_CAST(amount AS DOUBLE) >= 200 THEN 'high'
      WHEN TRY_CAST(amount AS DOUBLE) >= 50  THEN 'medium'
      WHEN TRY_CAST(amount AS DOUBLE) > 0    THEN 'low'
      ELSE                                        'invalid'
    END AS order_tier
  FROM bronze_orders
""").show(truncate=False)

## The Quarantine Pattern

Dropping bad records silently is dangerous — you lose data and have no way to investigate. The **quarantine pattern** routes invalid records to a separate table instead of discarding them.

```
Bronze table
     │
     ├── valid rows   ──▶ Silver table        (clean, typed, trustworthy)
     └── invalid rows ──▶ Quarantine table    (raw + rejection reason)
```

The quarantine table lets you:
- Audit data quality issues over time
- Fix upstream data problems with evidence
- Reprocess records once the source issue is resolved

Implementation: add a `rejection_reason` column using `CASE WHEN`, then split into two writes.

In [ ]:
# Tag each row with a rejection reason (null = valid)
spark.sql("""
  CREATE OR REPLACE TEMP VIEW tagged_orders AS
  SELECT
    *,
    CASE
      WHEN order_id IS NULL                            THEN 'missing order_id'
      WHEN customer_id IS NULL                         THEN 'missing customer_id'
      WHEN TRY_CAST(amount AS DOUBLE) IS NULL          THEN 'unparseable amount'
      WHEN TRY_CAST(amount AS DOUBLE) <= 0             THEN 'non-positive amount'
      WHEN COALESCE(
             TRY_TO_DATE(order_date,'yyyy-MM-dd'),
             TRY_TO_DATE(order_date,'dd/MM/yyyy')
           ) IS NULL                                   THEN 'unparseable date'
      ELSE NULL                                        -- valid
    END AS rejection_reason
  FROM (SELECT DISTINCT * FROM bronze_orders)
""")

spark.sql("SELECT order_id, amount, order_date, rejection_reason FROM tagged_orders").show(truncate=False)

In [ ]:
# Silver table — valid rows only, fully typed and cleaned
spark.sql("""
  CREATE OR REPLACE TABLE silver_orders
  USING DELTA AS
  SELECT
    order_id,
    customer_id,
    TRY_CAST(amount AS DOUBLE)                                         AS amount,
    COALESCE(
      TRY_TO_DATE(order_date, 'yyyy-MM-dd'),
      TRY_TO_DATE(order_date, 'dd/MM/yyyy')
    )                                                                  AS order_date,
    TRIM(region)                                                       AS region,
    LOWER(TRIM(email))                                                 AS email,
    CASE
      WHEN TRY_CAST(amount AS DOUBLE) >= 200 THEN 'high'
      WHEN TRY_CAST(amount AS DOUBLE) >= 50  THEN 'medium'
      ELSE                                        'low'
    END AS order_tier
  FROM tagged_orders
  WHERE rejection_reason IS NULL
""")

spark.sql("SELECT * FROM silver_orders").show(truncate=False)

In [ ]:
# Quarantine table — bad rows with rejection reason
spark.sql("""
  CREATE OR REPLACE TABLE quarantine_orders
  USING DELTA AS
  SELECT *, CURRENT_TIMESTAMP() AS quarantined_at
  FROM   tagged_orders
  WHERE  rejection_reason IS NOT NULL
""")

spark.sql("SELECT order_id, amount, order_date, rejection_reason FROM quarantine_orders").show(truncate=False)

In [ ]:
# Summary of the pipeline output
total     = spark.sql("SELECT COUNT(*) FROM bronze_orders").collect()[0][0]
silver_ct = spark.sql("SELECT COUNT(*) FROM silver_orders").collect()[0][0]
quar_ct   = spark.sql("SELECT COUNT(*) FROM quarantine_orders").collect()[0][0]

print(f"Bronze (raw)     : {total}")
print(f"Silver (clean)   : {silver_ct}")
print(f"Quarantine (bad) : {quar_ct}")

In [ ]:
# Cleanup
spark.sql("DROP SCHEMA IF EXISTS silver_demo CASCADE")

## Summary

- **Deduplication:** `DISTINCT` removes exact duplicates. `dropDuplicates(subset)` deduplicates by key. `ROW_NUMBER()` with a window gives deterministic dedup when you need to keep the latest or earliest record.
- **Null handling:** `NULL` and `''` are different — use `NULLIF(col, '')` to normalise empty strings to null before filtering. `COALESCE` returns the first non-null. `fillna`/`dropna` handle nulls in PySpark.
- **Type casting:** Use `TRY_CAST` (not `CAST`) on raw string data — it returns null on failure instead of throwing. Filter or quarantine the resulting nulls downstream.
- **String cleaning:** `TRIM` for whitespace, `LOWER`/`UPPER` for case, `REGEXP_REPLACE`/`REGEXP_EXTRACT` for patterns, `CONCAT_WS` for joining with a separator (skips nulls automatically).
- **Dates:** `TO_DATE(col, format)` and `TO_TIMESTAMP(col, format)` parse strings. Use `COALESCE(TRY_TO_DATE(..., fmt1), TRY_TO_DATE(..., fmt2))` for mixed formats.
- **Conditional logic:** `CASE WHEN` for multi-branch logic, `IF` for simple two-branch, `NULLIF` to convert a specific value to null, `COALESCE` for fallback chains.
- **Quarantine pattern:** tag bad rows with a `rejection_reason` column, write valid rows to Silver, write invalid rows to a separate quarantine table — never silently discard data.

Next: `07-sql-udfs-higher-order-functions.ipynb` — creating reusable SQL UDFs and working with arrays and maps using higher-order functions.